# 06 — Linear and Polynomial Regression
### From Supervised Learning Theory to Ridge Regularization

---

> **Position in the series:** This notebook is **Section 06**, following:
> `01_statistical_learning_theory` → `02_pac_and_vc_dimensions` → `03_linear_algebra_and_geometry` → `04_pca` → `05_gaussian_mixture_models` → **`06_linear_regression`** → `07_classification` ...

---

## References

| # | Source |
|---|--------|
| [1] | Andrew Ng — *CS229 Lecture Notes*, Stanford University (2019) |
| [2] | Iacopo Masi — *07_regression lecture slides & 11_regression_lsq_poly notebook*, Sapienza SMIA |
| [3] | C. M. Bishop — *Pattern Recognition and Machine Learning*, Springer, 2006 — Ch. 1, 3 |
| [4] | S. Shalev-Shwartz & S. Ben-David — *Understanding Machine Learning*, Cambridge, 2014 — Ch. 9 |
| [5] | G. H. Golub & C. F. Van Loan — *Matrix Computations*, 4th ed., Johns Hopkins, 2013 |

---

## Structure
1. [Supervised Learning Framework](#1)
2. [Generative Assumption: Gaussian Noise](#2)
3. [The Linear Model & Augmented Vector Trick](#3)
4. [Objective Function: Residual Sum of Squares](#4)
5. [From True Risk to ERM — Dimensional Analysis](#5)
6. [Three Roads to $\mathbf{w}^*$](#6)
   - 6A. Analytical: Normal Equations (full derivation)
   - 6B. Geometric: Orthogonal Projection & Hat Matrix
   - 6C. Probabilistic: MLE
7. [Iterative Optimization: Gradient Descent](#7)
8. [Polynomial Regression & Vandermonde Matrix](#8)
9. [Overfitting Diagnostics & Bias-Variance Trade-off](#9)
10. [Ridge Regularization — Spectral Analysis](#10)
11. [Expected Value of the Ridge Estimator](#11)
12. [Linear vs Polynomial vs Ridge — Full Comparison](#12)


---
## 1. Supervised Learning Framework

In the context of **Supervised Learning**, we operate with a training dataset consisting of $N$ pairs of inputs and their corresponding continuous targets:

$$\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$$

* **Input Space ($\mathcal{X}$):** Each input vector $\mathbf{x}_i \in \mathbb{R}^d$ represents a set of features.
* **Target Space ($\mathcal{Y}$):** Since we are dealing with regression, the target $y_i \in \mathbb{R}$ is a continuous scalar.
* **Hypothesis Space ($\mathcal{H}$):** The set of all possible functions $f: \mathcal{X} \rightarrow \mathcal{Y}$ our model can assume. Our goal is to find the optimal hypothesis $f^* \in \mathcal{H}$ that minimizes structural errors and generalizes to unseen data.

---

## 2. Fundamental Challenge: Underfitting vs. Overfitting

A core challenge is ensuring high **generalization capability** on unseen future data. This requires navigating the trade-off between two phenomena:

* **Underfitting (High Bias):** Occurs when $\mathcal{H}$ is too restrictive. The model fails to capture the underlying trend → high training error and high test error.
* **Overfitting (High Variance):** Occurs when the model has excessive capacity relative to available data. The model effectively *memorizes* training samples including their random noise → near-zero training error but massive test error due to wild oscillations between data points.

> **ML Bridge:** The tension between underfitting and overfitting is formalized by the **Bias-Variance decomposition** (see Section 9) and bounded in theory by the VC dimension (see `02_pac_and_vc_dimensions`).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline

np.random.seed(42)

# True underlying function
f_star = lambda x: np.sin(2 * np.pi * x)
N_train = 15
x_train = np.sort(np.random.uniform(0, 1, N_train))
y_train = f_star(x_train) + np.random.normal(0, 0.25, N_train)
x_plot  = np.linspace(0, 1, 300)

degrees = [1, 4, 14]
titles  = ['Underfitting\n(degree 1 — high bias)',
           'Good Fit\n(degree 4)',
           'Overfitting\n(degree 14 — high variance)']
colors  = ['#457B9D', '#2A9D8F', '#E63946']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, deg, title, color in zip(axes, degrees, titles, colors):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(x_train.reshape(-1,1), y_train)
    y_pred_tr   = model.predict(x_train.reshape(-1,1))
    y_pred_plot = model.predict(x_plot.reshape(-1,1))
    train_mse   = mean_squared_error(y_train, y_pred_tr)
    ax.scatter(x_train, y_train, s=50, color='black', zorder=5, label='Training data')
    ax.plot(x_plot, f_star(x_plot), 'k--', lw=2, alpha=0.6, label='$f^*$ (true)')
    ax.plot(x_plot, np.clip(y_pred_plot, -2.5, 2.5), color=color, lw=2.5, label=f'deg {deg}')
    ax.set_xlim(0, 1); ax.set_ylim(-2.5, 2.5)
    ax.set_title(f'{title}\nTrain MSE = {train_mse:.4f}', fontsize=10)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Supervised Learning: Underfitting — Good Fit — Overfitting', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 3. Generative Assumption: The Role of Gaussian Noise

In real-world scenarios, data is inherently imperfect. We model the relationship between the inputs and the true targets using a generative process:

$$y = f(\mathbf{x}) + \epsilon$$

Where $\epsilon$ represents the **intrinsic, irreducible noise** present within the data. We assume this noise follows a **Gaussian (Normal) Distribution** centered at zero with constant variance $\sigma^2$:

$$\epsilon \overset{\text{i.i.d.}}{\sim} \mathcal{N}(0, \sigma^2) \qquad p(\epsilon) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{\epsilon^2}{2\sigma^2}\right)$$

This implies that each target is conditionally Gaussian:

$$y_i \mid \mathbf{x}_i \sim \mathcal{N}(f(\mathbf{x}_i),\, \sigma^2)$$

### Key Clarification on Variance and Overfitting

The variance $\sigma^2$ belongs to the environment's noise and is completely **aleatoric** (irreducible). The model *cannot* learn this noise because its expected value is zero: $\mathbb{E}[\epsilon]=0$.

**Overfitting does not happen because we study $\sigma^2$**; rather, overfitting occurs because a high-capacity model tries to minimize the training error to absolute zero, shaping its parameters around the *specific random instances* of $\epsilon$ in the training set instead of learning the true deterministic function $f(\mathbf{x})$.

> **Connection to PCA (04):** Gaussian noise is the same noise model used in Probabilistic PCA, where the latent variable model is $\mathbf{x} = \mathbf{W}\mathbf{z} + \boldsymbol{\mu} + \boldsymbol{\epsilon}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(7)
w_true  = np.array([1.5, 3.0])
x_line  = np.linspace(0, 2, 200)
f_line  = w_true[0] + w_true[1] * x_line

sigma_values = [0.15, 0.5, 1.2]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

for ax, sigma in zip(axes, sigma_values):
    x_tr = np.random.uniform(0, 2, 25)
    y_tr = w_true[0] + w_true[1]*x_tr + np.random.normal(0, sigma, 25)
    ax.plot(x_line, f_line, 'k--', lw=2.2, label='$f^*(x) = w_0 + w_1 x$')
    ax.scatter(x_tr, y_tr, s=45, color='#E63946', zorder=5, label='Noisy observations')
    # Gaussian tube at two query points
    for x0 in [0.7, 1.6]:
        mu = w_true[0] + w_true[1]*x0
        eps_r   = np.linspace(mu - 3.5*sigma, mu + 3.5*sigma, 300)
        pdf     = norm.pdf(eps_r, mu, sigma)
        pdf_s   = pdf / pdf.max() * 0.28
        ax.fill_betweenx(eps_r, x0, x0+pdf_s, alpha=0.25, color='#457B9D')
        ax.plot(x0+pdf_s, eps_r, color='#457B9D', lw=1.5)
        ax.axvline(x0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'$\\sigma = {sigma}$ — noise level', fontsize=11)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle(
    'Generative Model $y = f(\\mathbf{x}) + \\epsilon$,  '
    '$\\epsilon \\sim \\mathcal{N}(0,\\sigma^2)$\n'
    'Noise is aleatoric (irreducible) — overfitting comes from fitting it, not from $\\sigma^2$ itself',
    fontsize=11)
plt.tight_layout(); plt.show()


---
## 4. The Linear Regression Model and the Augmented Vector Trick

The simplest parametric hypothesis for regression is the **Linear Regression** model. Geometrically, the mapping $\hat{y} = \mathbf{w}^T\mathbf{x}$ forms a hyperplane constrained to pass strictly through the origin $(0,0,\dots,0)$ of the vector space.

To overcome this geometric limitation and create an **affine** hyperplane, we introduce a shift parameter known as the **Bias** $w_0$:

$$\hat{y} = w_0 + w_1x_1 + w_2x_2 + \dots + w_dx_d = w_0 + \sum_{j=1}^d w_j x_j$$

### The Augmented Vector Trick

To avoid dealing with the bias as an isolated term during optimization, we apply the **augmented vector trick**: prepend a constant feature $x_0 = 1$ to every input, and prepend $w_0$ to the parameter vector:

$$\mathbf{x}_{\text{aug}} = \begin{bmatrix} 1 \\ x_1 \\ \vdots \\ x_d \end{bmatrix} \in \mathbb{R}^{d+1}, \qquad \mathbf{w}_{\text{aug}} = \begin{bmatrix} w_0 \\ w_1 \\ \vdots \\ w_d \end{bmatrix} \in \mathbb{R}^{d+1}$$

This allows us to express the entire model as a single dot product:

$$\boxed{\hat{y} = \mathbf{w}^T\mathbf{x}}$$

The full dataset is stacked into the **Design Matrix** $\mathbf{X} \in \mathbb{R}^{N \times (d+1)}$:

$$\mathbf{X} = \begin{bmatrix} —\, \mathbf{x}_1^T\, — \\ —\, \mathbf{x}_2^T\, — \\ \vdots \\ —\, \mathbf{x}_N^T\, — \end{bmatrix}, \qquad \hat{\mathbf{y}} = \mathbf{X}\mathbf{w}$$

The first column of $\mathbf{X}$ is entirely ones — encoding the bias for every sample simultaneously.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: bias vs no-bias in 1D
x1d = np.linspace(-1, 3, 200)
ax  = axes[0]
ax.plot(x1d, 2.0*x1d,        '#E63946', lw=2.5, label='$\\hat{y}=wx$  (through origin)')
ax.plot(x1d, 1.5+2.0*x1d,   '#2A9D8F', lw=2.5, label='$\\hat{y}=w_0+wx$  (affine)')
ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
ax.scatter([0],[0],   s=120, color='#E63946', zorder=5)
ax.scatter([0],[1.5], s=120, color='#2A9D8F', zorder=5)
ax.annotate('constrained to origin', xy=(0,0),   xytext=(0.25,-0.7),  fontsize=8.5, color='#E63946', arrowprops=dict(arrowstyle='->', color='#E63946', lw=1.2))
ax.annotate('free intercept $w_0=1.5$', xy=(0,1.5), xytext=(0.25,2.0), fontsize=8.5, color='#2A9D8F', arrowprops=dict(arrowstyle='->', color='#2A9D8F', lw=1.2))
ax.set_title('Geometric Effect of the Bias Term', fontsize=11)
ax.set_xlabel('$x$'); ax.set_ylabel('$\\hat{y}$')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── Right: design matrix table
ax2 = axes[1]; ax2.axis('off')
table_data = [
    ['', '$x_0=1$', '$x_1$', '$x_2$', '$\\cdots$', '$x_d$'],
    ['$\\mathbf{x}_1^T$', '1', '$x_{11}$', '$x_{12}$', '$\\cdots$', '$x_{1d}$'],
    ['$\\mathbf{x}_2^T$', '1', '$x_{21}$', '$x_{22}$', '$\\cdots$', '$x_{2d}$'],
    ['$\\vdots$',         '$\\vdots$', '$\\vdots$', '$\\vdots$', '$\\ddots$', '$\\vdots$'],
    ['$\\mathbf{x}_N^T$', '1', '$x_{N1}$', '$x_{N2}$', '$\\cdots$', '$x_{Nd}$'],
]
tbl = ax2.table(cellText=table_data, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10.5); tbl.scale(1.5, 2.0)
ax2.set_title('Design Matrix $\\mathbf{X} \\in \\mathbb{R}^{N \\times (d+1)}$\n(first column = bias ones)', fontsize=11)

plt.suptitle('The Augmented Vector Trick', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Numerical verification
print('=== Augmented Vector Trick — Numerical Verification ===')
N, d = 5, 2
X_raw = np.array([[1.2,0.5],[2.1,-0.3],[0.8,1.1],[3.0,0.0],[1.5,-0.9]])
X_aug = np.hstack([np.ones((N,1)), X_raw])
w     = np.array([0.5, 1.2, -0.8])
print(f'X_raw (N={N}, d={d}):\n{X_raw}')
print(f'\nX_aug (N={N}, d+1={d+1}): prepend ones column\n{X_aug}')
print(f'\nw = {w}')
print(f'y_hat = X_aug @ w = {(X_aug @ w).round(4)}')


---
## 5. Objective Function: Residual Sum of Squares (RSS)

To evaluate our parameters, we define a **Loss Function** measuring the discrepancy between ground-truth targets $\mathbf{y}$ and predictions $\hat{\mathbf{y}} = \mathbf{X}\mathbf{w}$. For linear regression we use the **Residual Sum of Squares (RSS)**:

$$\boxed{L(\mathbf{w}) = \text{RSS}(\mathbf{w}) = \sum_{i=1}^N (y_i - \mathbf{w}^T\mathbf{x}_i)^2 = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 = (\mathbf{y}-\mathbf{X}\mathbf{w})^T(\mathbf{y}-\mathbf{X}\mathbf{w})}$$

### Why the $L_2$ Squared Norm?

1. **Mathematical Optimization:** The squared $L_2$ norm yields a **convex, continuous, and differentiable** objective. Unlike the $L_1$ norm (absolute error), which has a non-differentiable kink at zero, $L_2$ can be differentiated everywhere — the global minimum is found via calculus.
2. **Statistical Justification:** Under i.i.d. Gaussian noise, **MLE is mathematically equivalent to minimizing RSS** (derived in Section 6C).

### The Algebraic Convenience of the $\frac{1}{2}$ Factor

We often scale the objective by $\frac{1}{2}$ (or $\frac{1}{2N}$). Multiplying by a positive scalar does not shift the $\arg\min$:

$$L(\mathbf{w}) = \frac{1}{2} \sum_{i=1}^N (y_i - \mathbf{w}^T\mathbf{x}_i)^2$$

When differentiating, the exponent $2$ pulls forward and cancels with $\frac{1}{2}$, yielding a clean gradient without extra constants.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(3)
x_data = np.array([1., 2., 3., 4., 5.])
y_data = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

# RSS loss surface
w0_r = np.linspace(-2, 5, 120)
w1_r = np.linspace(-1, 4, 120)
W0, W1 = np.meshgrid(w0_r, w1_r)
X_aug  = np.column_stack([np.ones_like(x_data), x_data])
RSS    = np.array([
    np.sum((y_data - W0[i,j] - W1[i,j]*x_data)**2)
    for i in range(120) for j in range(120)
]).reshape(120, 120)
w_opt = np.linalg.lstsq(X_aug, y_data, rcond=None)[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Contour
ax = axes[0]
cp = ax.contourf(W0, W1, RSS, levels=45, cmap='RdYlGn_r', alpha=0.88)
plt.colorbar(cp, ax=ax, label='RSS')
ax.contour(W0, W1, RSS, levels=22, colors='white', linewidths=0.5, alpha=0.35)
ax.scatter(*w_opt, s=250, color='blue', marker='*', zorder=10,
           label=f'$\\mathbf{{w}}^*=({w_opt[0]:.2f},{w_opt[1]:.2f})$')
ax.set_xlabel('$w_0$ (bias)', fontsize=11); ax.set_ylabel('$w_1$ (slope)', fontsize=11)
ax.set_title('RSS Loss Surface — Contour View\n(unique global minimum: convex function)', fontsize=10)
ax.legend(fontsize=9)

# Residuals
ax2 = axes[1]
x_plot = np.linspace(0, 6, 200)
ax2.scatter(x_data, y_data, s=90, color='black', zorder=5, label='Data')
ax2.plot(x_plot, w_opt[0]+w_opt[1]*x_plot, '#E63946', lw=2.5,
         label=f'$\\hat{{y}}={w_opt[0]:.2f}+{w_opt[1]:.2f}x$')
for xi, yi in zip(x_data, y_data):
    yi_hat = w_opt[0]+w_opt[1]*xi
    ax2.plot([xi,xi],[yi,yi_hat], '#457B9D', lw=2, linestyle='--')
    ax2.annotate(f'$r_{int(xi)}$', xy=(xi+0.07,(yi+yi_hat)/2), fontsize=9, color='#457B9D')
ax2.set_xlabel('$x$'); ax2.set_ylabel('$y$')
ax2.set_title('Residuals = vertical distances to regression line\n$r_i = y_i - \\hat{y}_i$ (NOT perpendicular to line)', fontsize=10)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.suptitle(r'$L(\mathbf{w}) = \|\mathbf{y}-\mathbf{X}\mathbf{w}\|_2^2$: Convex Loss and Residuals', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Optimal: w0={w_opt[0]:.4f}, w1={w_opt[1]:.4f}  |  Min RSS={np.sum((y_data-X_aug@w_opt)**2):.4f}')


---
## 5. From True Risk to Empirical Risk Minimization (ERM) — Dimensional Analysis

To justify why we minimize the sum of squared errors, we look through the lens of **Statistical Learning Theory**:

### True Risk (Generalization Error)

The expected loss of $f$ over the *entire* theoretical population under the true joint distribution $p(\mathbf{x},y)$:

$$R(f) = \mathbb{E}_{(\mathbf{x},y)\sim p}[L(y,f(\mathbf{x}))] = \iint L(y,f(\mathbf{x}))\,p(\mathbf{x},y)\,d\mathbf{x}\,dy$$

Because $p(\mathbf{x},y)$ is unknown, this integral is **completely uncomputable**.

### Empirical Risk Minimization (ERM)

By the Law of Large Numbers, we approximate the expectation with a finite-sample mean:

$$\hat{R}_{\text{emp}}(\mathbf{w}) = \frac{1}{N}\sum_{i=1}^N L(y_i,f(\mathbf{x}_i)) \xrightarrow{N\to\infty} R(f)$$

For the squared loss, ERM is exactly **minimizing the MSE (or RSS)**. The ERM framework is the theoretical justification for fitting $\mathbf{w}^*$.

---

### Rigorous Dimensional Analysis

Before any matrix computation, always verify dimensions:

| Object | Dimension | Role |
|--------|-----------|------|
| $\mathbf{X}$ | $N \times (d+1)$ | Design matrix |
| $\mathbf{w}$ | $(d+1) \times 1$ | Weight vector |
| $\mathbf{y}$ | $N \times 1$ | Target vector |
| $\mathbf{X}\mathbf{w}$ | $N \times 1$ | Predictions |
| $\mathbf{y} - \mathbf{X}\mathbf{w}$ | $N \times 1$ | Residual vector $\mathbf{r}$ |
| $\mathbf{r}^T\mathbf{r} = L(\mathbf{w})$ | scalar | Loss |
| $\mathbf{X}^T\mathbf{X}$ | $(d+1)\times(d+1)$ | Empirical covariance |
| $\mathbf{X}^T\mathbf{y}$ | $(d+1)\times 1$ | Cross-covariance |


In [ ]:
import numpy as np

np.random.seed(1)
N, d = 50, 3
X = np.hstack([np.ones((N,1)), np.random.randn(N,d)])
w_true = np.array([1.0, 2.5, -1.2, 0.8])
y = X @ w_true + np.random.normal(0, 0.5, N)

print('=== Dimensional Analysis ===')
print(f'N={N} samples, d={d} features')
for name, obj in [
    ('X          ', X),
    ('w          ', w_true),
    ('y          ', y),
    ('Xw         ', X @ w_true),
    ('r = y - Xw ', y - X @ w_true),
    ('X^T X      ', X.T @ X),
    ('X^T y      ', X.T @ y),
]:
    obj_arr = np.atleast_2d(obj) if obj.ndim == 1 else obj
    print(f'  {name}: shape {obj.shape}')

# ERM in action
def mse(w, X, y):
    r = y - X @ w
    return (r @ r) / len(y)

w_rand = np.random.randn(d+1)
w_opt  = np.linalg.lstsq(X, y, rcond=None)[0]

print(f'\n=== Empirical Risk (MSE) ===')
print(f'MSE (random w):   {mse(w_rand, X, y):.4f}')
print(f'MSE (optimal w*): {mse(w_opt,  X, y):.6f}')
print(f'\nTrue w:  {w_true}')
print(f'Found w*:{w_opt.round(4)}')
print(f'Error:   {np.abs(w_opt - w_true).round(4)}')


---
## 6. Three Roads to the Optimal Parameters ($\mathbf{w}^*$)

$$\mathbf{w}^* = \arg\min_{\mathbf{w}} L(\mathbf{w}) = \arg\min_{\mathbf{w}} \|\mathbf{y}-\mathbf{X}\mathbf{w}\|_2^2$$

Three complementary frameworks lead to the *same* closed-form solution.

---

### 6A. The Analytical Approach — Full Derivation of the Normal Equations

Because the RSS is convex, we compute its gradient, set it to zero, and solve:

**Step 1 — Expand the loss:**

$$L(\mathbf{w}) = (\mathbf{y}-\mathbf{X}\mathbf{w})^T(\mathbf{y}-\mathbf{X}\mathbf{w}) = \mathbf{y}^T\mathbf{y} - 2\mathbf{w}^T\mathbf{X}^T\mathbf{y} + \mathbf{w}^T\mathbf{X}^T\mathbf{X}\mathbf{w}$$

**Step 2 — Take the gradient** (using $\nabla_\mathbf{w}\mathbf{b}^T\mathbf{w}=\mathbf{b}$ and $\nabla_\mathbf{w}\mathbf{w}^TA\mathbf{w}=2A\mathbf{w}$ for symmetric $A$):

$$\nabla_\mathbf{w} L(\mathbf{w}) = -2\mathbf{X}^T\mathbf{y} + 2\mathbf{X}^T\mathbf{X}\mathbf{w} = -2\mathbf{X}^T(\mathbf{y}-\mathbf{X}\mathbf{w})$$

**Step 3 — Set to zero:**

$$\nabla_\mathbf{w} L(\mathbf{w}) = \mathbf{0} \implies \underbrace{\mathbf{X}^T\mathbf{X}\,\mathbf{w} = \mathbf{X}^T\mathbf{y}}_{\text{Normal Equations}}$$

**Step 4 — Solve** (assuming $\mathbf{X}^T\mathbf{X}$ is invertible):

$$\boxed{\mathbf{w}^* = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}}$$

This closed-form solution provides the exact minimum in a single computational step at cost $\mathcal{O}(d^3)$.

> **Invertibility condition:** $\mathbf{X}^T\mathbf{X}$ is invertible if and only if $\text{rank}(\mathbf{X})=d+1$, i.e., the columns of $\mathbf{X}$ are linearly independent and $N \geq d+1$. Collinear features or $N < d+1$ make the system singular.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(5)
N, d = 60, 3
X = np.hstack([np.ones((N,1)), np.random.randn(N,d)])
w_true = np.array([1.5, 2.0, -1.0, 0.5])
y = X @ w_true + np.random.normal(0, 0.6, N)

XtX  = X.T @ X
Xty  = X.T @ y
w_ne = np.linalg.inv(XtX) @ Xty             # Normal Equations
w_ls = np.linalg.lstsq(X, y, rcond=None)[0]  # numerically stable (QR)

print('=== Normal Equations: w* = (X^T X)^{-1} X^T y ===')
print(f'X^T X  shape: {XtX.shape}  (empirical covariance)')
print(f'X^T y  shape: {Xty.shape}')
print(f'\nw* (Normal Eq.): {w_ne.round(6)}')
print(f'w* (lstsq/QR):   {w_ls.round(6)}')
print(f'True w:          {w_true}')
print(f'Identical:       {np.allclose(w_ne, w_ls, atol=1e-8)}')

grad_at_w = -2*X.T@y + 2*XtX@w_ne
print(f'\n||grad L(w*)|| = {np.linalg.norm(grad_at_w):.2e}   (must be ~0 at minimum)')

# Convexity: loss along a 1D slice through w*
t_vals = np.linspace(-3, 3, 300)
d_vec  = np.random.randn(d+1); d_vec /= np.linalg.norm(d_vec)
losses = [np.sum((y - X@(w_ne+t*d_vec))**2) for t in t_vals]

plt.figure(figsize=(7,4))
plt.plot(t_vals, losses, '#E63946', lw=2.5)
plt.axvline(0, color='black', lw=2, linestyle='--', label='$\\mathbf{w}^*$ — unique global minimum')
plt.xlabel('Step $t$ along a random direction', fontsize=11)
plt.ylabel('$L(\\mathbf{w}^*+t\\mathbf{d})$', fontsize=11)
plt.title('RSS is strictly convex:\n$\\mathbf{w}^*$ is the unique global minimum in every direction', fontsize=11)
plt.legend(fontsize=10); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
### 6B. The Geometric Approach — Orthogonal Projections & the Hat Matrix

#### The Geometry of Residuals: Data Space vs. Vector Space

Understanding Least Squares geometrically requires distinguishing two different spaces:

* **In the Data Space (2D/3D feature plot):** Residuals are the **vertical distances** (parallel to the $Y$-axis) between $y_i$ and $\hat{y}_i$. They are *not* the perpendicular distance to the tilted regression line — because our objective strictly minimizes error along the target dimension $\mathcal{Y}$.

* **In the Column Space ($\mathbb{R}^N$):** The target vector $\mathbf{y}\in\mathbb{R}^N$ exists in an $N$-dimensional space. The columns of $\mathbf{X}$ span a lower-dimensional subspace $\text{span}(\mathbf{X})$. Since $\mathbf{y}$ generally does not lie within $\text{span}(\mathbf{X})$, we find its best approximation via **orthogonal projection** onto this subspace.

#### Overdetermined Systems

In practice we almost always have $N \gg d$: more equations than unknowns. The system $\mathbf{X}\mathbf{w}=\mathbf{y}$ has no exact solution. OLS provides the optimal compromise: the solution that minimizes $\|\mathbf{y}-\mathbf{X}\mathbf{w}\|_2^2$.

The residual $\mathbf{r}=\mathbf{y}-\mathbf{X}\mathbf{w}^*$ is minimized when it is **orthogonal** to the column space:

$$\mathbf{X}^T(\mathbf{y}-\mathbf{X}\mathbf{w}^*) = \mathbf{0} \implies \mathbf{X}^T\mathbf{X}\mathbf{w}^*=\mathbf{X}^T\mathbf{y} \quad \text{(Normal Equations again)}$$

#### The Projection Matrix (Hat Matrix)

Substituting $\mathbf{w}^*=(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ into $\hat{\mathbf{y}}=\mathbf{X}\mathbf{w}^*$:

$$\hat{\mathbf{y}} = \mathbf{X}(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y} =: \mathbf{H}\mathbf{y}$$

The **Hat Matrix** $\mathbf{H}=\mathbf{X}(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T$ *puts a hat on* $\mathbf{y}$:

$$\hat{\mathbf{y}} = \mathbf{H}\mathbf{y}, \qquad \mathbf{r} = (\mathbf{I}-\mathbf{H})\mathbf{y}$$

**Fundamental algebraic properties:**
1. **Symmetry:** $\mathbf{H}^T = \mathbf{H}$
2. **Idempotency:** $\mathbf{H}^2=\mathbf{H}$ — projecting a vector already in the subspace changes nothing
3. **$(\mathbf{I}-\mathbf{H})$ is also symmetric and idempotent** — it projects onto the orthogonal complement
4. **Eigenvalues of $\mathbf{H}$:** only $0$ or $1$; rank$(\mathbf{H})=d+1$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm as spnorm

np.random.seed(9)
N, d = 40, 2
X = np.hstack([np.ones((N,1)), np.random.randn(N,d)])
w_true = np.array([1.0, 2.0, -0.5])
y = X @ w_true + np.random.normal(0, 0.5, N)

H     = X @ np.linalg.inv(X.T @ X) @ X.T
y_hat = H @ y
r     = y - y_hat

print('=== Hat Matrix Properties ===')
print(f'Shape: {H.shape}')
print(f'Symmetric:        H == H^T:     {np.allclose(H, H.T)}')
print(f'Idempotent:       H^2 == H:     {np.allclose(H@H, H)}')
print(f'(I-H) idempotent:              {np.allclose((np.eye(N)-H)@(np.eye(N)-H), np.eye(N)-H)}')
print(f'Orthogonality:    max|X^T r| = {np.abs(X.T@r).max():.2e}   (must be ~0)')

eigs = np.sort(np.linalg.eigvalsh(H))[::-1]
n1   = np.sum(np.isclose(eigs, 1., atol=1e-8))
n0   = np.sum(np.isclose(eigs, 0., atol=1e-8))
print(f'Eigenvalues of H: {n1} ones, {n0} zeros  (rank = d+1 = {d+1})')
print(f'trace(H) = rank(H) = {H.trace():.4f}  (= d+1 = {d+1})')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

# Data space: residuals as vertical distances
ax = axes[0]
x1d = X[:,1]
w1d = np.linalg.lstsq(np.column_stack([np.ones(N), x1d]), y, rcond=None)[0]
xs  = np.linspace(x1d.min()-.3, x1d.max()+.3, 200)
ax.scatter(x1d, y, s=40, color='black', zorder=5, label='Data')
ax.plot(xs, w1d[0]+w1d[1]*xs, '#E63946', lw=2.5, label='$\\hat{y}$ (fit)')
for xi, yi in zip(x1d, y):
    ax.plot([xi,xi],[yi, w1d[0]+w1d[1]*xi], '#457B9D', lw=1.0, alpha=0.65)
ax.set_title('Data Space:\nvertical residuals (parallel to $Y$-axis)', fontsize=10)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$y$')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Residual distribution
ax2 = axes[1]
ax2.hist(r, bins=14, color='#2A9D8F', edgecolor='white', alpha=0.85, density=True)
xs2 = np.linspace(r.min()-.5, r.max()+.5, 200)
ax2.plot(xs2, spnorm.pdf(xs2, r.mean(), r.std()), '#E63946', lw=2.5, label='Fitted $\\mathcal{N}$')
ax2.axvline(0, color='black', lw=2, linestyle='--', label=f'Mean={r.mean():.4f}')
ax2.set_xlabel('Residual $r_i$', fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.set_title(f'Residual distribution\nmean={r.mean():.4f}, std={r.std():.4f}', fontsize=10)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# Hat matrix heatmap
ax3 = axes[2]
im = ax3.imshow(H[:20,:20], cmap='RdBu_r', aspect='auto', vmin=-0.2, vmax=0.2)
plt.colorbar(im, ax=ax3)
ax3.set_title('Hat Matrix $\\mathbf{H}$ (first 20x20)\nSymmetric projection operator', fontsize=10)
ax3.set_xlabel('Sample index'); ax3.set_ylabel('Sample index')

plt.suptitle('Hat Matrix $\\mathbf{H}=\\mathbf{X}(\\mathbf{X}^T\\mathbf{X})^{-1}\\mathbf{X}^T$: Geometry of Projection', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


---
### 6C. The Probabilistic Approach — Maximum Likelihood Estimation (MLE)

Modeling $y_i \sim \mathcal{N}(\mathbf{w}^T\mathbf{x}_i, \sigma^2)$, the conditional density of a single target:

$$p(y_i \mid \mathbf{x}_i;\,\mathbf{w},\sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(y_i-\mathbf{w}^T\mathbf{x}_i)^2}{2\sigma^2}\right)$$

By the i.i.d. assumption, the **Likelihood Function** for the full dataset:

$$L(\mathbf{w}) = \prod_{i=1}^N p(y_i\mid\mathbf{x}_i;\,\mathbf{w},\sigma^2)$$

### The Log-Likelihood Transformation

Maximizing a product of exponentials is numerically unstable (underflow). Since $\log$ is monotonically increasing, $\arg\max L = \arg\max \log L$:

$$\ln L(\mathbf{w}) = \underbrace{-\frac{N}{2}\log(2\pi\sigma^2)}_{\text{constant w.r.t. }\mathbf{w}} - \frac{1}{2\sigma^2}\underbrace{\sum_{i=1}^N(y_i-\mathbf{w}^T\mathbf{x}_i)^2}_{\text{RSS}(\mathbf{w})}$$

### From MLE to Least Squares

Maximizing over $\mathbf{w}$ drops the constant, and maximizing a negative quantity equals minimizing its positive counterpart:

$$\mathbf{w}^*_{\text{MLE}} = \arg\max_\mathbf{w}\ln L(\mathbf{w}) = \arg\min_\mathbf{w}\,\frac{1}{2}\sum_{i=1}^N(y_i-\mathbf{w}^T\mathbf{x}_i)^2 = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

Under Gaussian noise, **MLE converges to the exact same solution as the geometric Least Squares** — not a coincidence, but a deep structural result.

> **Bayesian Extension:** Introducing a prior $p(\mathbf{w})$ transitions us to **MAP estimation**. A Gaussian prior $\mathbf{w}\sim\mathcal{N}(\mathbf{0},\frac{1}{\lambda}\mathbf{I})$ yields Ridge regression. A Laplace prior yields Lasso. This is the probabilistic foundation of regularization.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(11)
N = 40
x = np.sort(np.random.uniform(0, 4, N))
w_true = np.array([0.5, 1.8])
sigma  = 0.6
y = w_true[0] + w_true[1]*x + np.random.normal(0, sigma, N)

X    = np.column_stack([np.ones(N), x])
w_mle = np.linalg.inv(X.T@X) @ X.T @ y

# Log-likelihood as function of w1 (fix w0 at MLE)
w1_range = np.linspace(0.5, 3.5, 400)
log_lik  = np.array([
    -N/2*np.log(2*np.pi*sigma**2) - 1/(2*sigma**2)*np.sum((y - w_mle[0] - w1*x)**2)
    for w1 in w1_range
])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(w1_range, log_lik, '#2A9D8F', lw=2.5, label='$\\ell(w_1)$ — log-likelihood')
ax.axvline(w_mle[1], color='#E63946', lw=2.5, linestyle='--',
           label=f'$w_1^*_{{\\mathrm{{MLE}}}}={w_mle[1]:.4f}$')
ax.axvline(w_true[1], color='gray', lw=1.8, linestyle=':',
           label=f'True $w_1={w_true[1]}$')
ax.set_xlabel('$w_1$', fontsize=11); ax.set_ylabel('$\\ell(\\mathbf{w})$', fontsize=11)
ax.set_title('Maximizing Log-Likelihood $\\equiv$ Minimizing RSS', fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax2 = axes[1]
x_plot = np.linspace(-0.5, 4.5, 300)
y_fit  = w_mle[0] + w_mle[1]*x_plot
ax2.fill_between(x_plot, y_fit-2*sigma, y_fit+2*sigma, alpha=0.10, color='#457B9D', label='$\\pm2\\sigma$')
ax2.fill_between(x_plot, y_fit-sigma,   y_fit+sigma,   alpha=0.20, color='#457B9D', label='$\\pm\\sigma$')
ax2.plot(x_plot, y_fit, '#E63946', lw=2.5, label=f'$\\hat{{y}}={w_mle[0]:.2f}+{w_mle[1]:.2f}x$')
ax2.scatter(x, y, s=50, color='black', zorder=5, label='Data')
ax2.set_xlabel('$x$'); ax2.set_ylabel('$y$')
ax2.set_title('MLE Fit with Gaussian Uncertainty Bands\n($\\pm\\sigma$ and $\\pm2\\sigma$)', fontsize=11)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.suptitle('Probabilistic View: MLE $\\equiv$ Least Squares under Gaussian Noise', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'MLE solution:  w0={w_mle[0]:.4f}, w1={w_mle[1]:.4f}')
print(f'True weights:  w0={w_true[0]:.4f}, w1={w_true[1]:.4f}')


---
## 7. Iterative Optimization: Gradient Descent

While the Normal Equations provide an exact closed-form solution, they present significant computational bottlenecks at scale:

| Operation | Cost |
|-----------|------|
| $\mathbf{X}^T\mathbf{X}$ | $\mathcal{O}(d^2 N)$ |
| $(\mathbf{X}^T\mathbf{X})^{-1}$ | $\mathcal{O}(d^3)$ (standard) or $\mathcal{O}(d^{2.373})$ (Coppersmith-Winograd) |
| Memory | $\mathcal{O}(Nd)$ for design matrix |

When $d$ reaches tens of thousands (NLP, genomics), inversion becomes **computationally prohibitive and numerically unstable**.

### Mathematical Formulation — The Update Rule

The gradient $\nabla_\mathbf{w} L$ points in the direction of **steepest ascent**. To minimize, move opposite:

$$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta\,\nabla_\mathbf{w} L(\mathbf{w}^{(t)})$$

### Gradient of the MSE Loss

For $L(\mathbf{w}) = \frac{1}{2N}\|\mathbf{X}\mathbf{w}-\mathbf{y}\|_2^2$:

$$\nabla_\mathbf{w} L(\mathbf{w}) = \frac{1}{N}\mathbf{X}^T(\mathbf{X}\mathbf{w}-\mathbf{y})$$

$$\boxed{\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \frac{\eta}{N}\,\mathbf{X}^T(\mathbf{X}\mathbf{w}^{(t)}-\mathbf{y})}$$

$(\mathbf{X}\mathbf{w}^{(t)}-\mathbf{y})$ computes the prediction error; $\mathbf{X}^T$ projects it back onto the feature dimensions, attributing the error to each weight.

For **Ridge Regression**, the gradient includes the $L_2$ penalty derivative:

$$\nabla_\mathbf{w} L_{\text{Ridge}}(\mathbf{w}) = \frac{1}{N}\mathbf{X}^T(\mathbf{X}\mathbf{w}-\mathbf{y}) + \lambda\mathbf{w}$$

### The Critical Role of the Learning Rate ($\eta$)

* **Too small:** Microscopic steps — stable but requires enormous iterations.
* **Just right:** Smooth convergence to $\mathbf{w}^*$.
* **Too large:** Oscillations; loss diverges to $+\infty$.

Since the OLS objective is **strictly convex**, any sufficiently small $\eta$ guarantees convergence to the unique global minimum — the same solution as the Normal Equations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
N = 100
X_raw = np.random.rand(N, 1)
y     = 3.0*X_raw.squeeze() + 2.0 + np.random.normal(0, 0.15, N)
X     = np.hstack([np.ones((N,1)), X_raw])

def gradient_descent(X, y, lr, n_iter=800):
    w = np.zeros(X.shape[1])
    losses, ws = [], []
    for _ in range(n_iter):
        r    = X @ w - y
        loss = (1/(2*len(y))) * (r@r)
        grad = (1/len(y)) * (X.T @ r)
        w    = w - lr*grad
        losses.append(loss); ws.append(w.copy())
    return np.array(ws), np.array(losses)

w_ne     = np.linalg.inv(X.T@X) @ X.T @ y
min_loss = (1/(2*N))*np.sum((X@w_ne - y)**2)

lrs    = [0.005, 0.5, 1.95]
labels = ['$\\eta=0.005$ (too small)', '$\\eta=0.5$ (good)', '$\\eta=1.95$ (too large)']
colors_lr = ['#457B9D', '#2A9D8F', '#E63946']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for lr, label, color, ax in zip(lrs, labels, colors_lr, axes):
    _, losses = gradient_descent(X, y, lr)
    ax.plot(np.clip(losses, 0, 5), color=color, lw=2.2)
    ax.axhline(min_loss, color='black', lw=1.8, linestyle='--', label='Normal Eq. minimum')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Iteration'); ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=8.5); ax.grid(True, alpha=0.3); ax.set_ylim(0, 5)

plt.suptitle('Gradient Descent: Effect of Learning Rate $\\eta$', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Convergence table
ws_good, losses_good = gradient_descent(X, y, lr=0.5, n_iter=600)
w_final = ws_good[-1]
print('=== From-Scratch Gradient Descent (eta=0.5) ===')
print(f'{"Iter":>6} | {"w0":>10} | {"w1":>10} | {"MSE Loss":>12}')
print('-'*48)
for t in [0, 20, 100, 200, 400, 599]:
    print(f'{t:>6} | {ws_good[t,0]:>10.6f} | {ws_good[t,1]:>10.6f} | {losses_good[t]:>12.8f}')
print(f'\nNormal Eq: w0={w_ne[0]:.6f}, w1={w_ne[1]:.6f}')
print(f'GD final:  w0={w_final[0]:.6f}, w1={w_final[1]:.6f}')
print(f'Converged: {np.allclose(w_final, w_ne, atol=1e-3)}')


---
## 8. Polynomial Regression & the Vandermonde Matrix

### Conceptual Motivation

In simple linear regression, we assume $y = w_0 + w_1 x + \epsilon$. When real-world data exhibits a curved trend, this model suffers from **Underfitting** — the class of linear functions is too rigid. To overcome this without losing OLS efficiency, we introduce **Polynomial Regression**:

$$y = w_0 + w_1 x + w_2 x^2 + \dots + w_d x^d + \epsilon = w_0 + \sum_{j=1}^d w_j x^j + \epsilon$$

### The Principle of Linearity in Parameters

Although the mapping is **nonlinear in $x$**, it is **strictly linear in $\mathbf{w}$**. The coefficients are never raised to a power or passed into nonlinear functions. This means the entire problem is recast as multivariate linear regression in an expanded feature space — and the Normal Equations still apply.

### The Vandermonde Matrix

Define the feature map $\phi: \mathbb{R} \to \mathbb{R}^{d+1}$:

$$\phi(x) = \begin{bmatrix} 1 & x & x^2 & \cdots & x^d \end{bmatrix}$$

Applying $\phi$ to all $n$ samples yields the **Vandermonde Matrix** $\mathbf{\Phi} \in \mathbb{R}^{n\times(d+1)}$:

$$\mathbf{\Phi} = \begin{bmatrix} 1 & x_1 & x_1^2 & \cdots & x_1^d \\ 1 & x_2 & x_2^2 & \cdots & x_2^d \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & x_n & x_n^2 & \cdots & x_n^d \end{bmatrix}$$

The system becomes $\mathbf{y}=\mathbf{\Phi}\mathbf{w}+\boldsymbol{\epsilon}$, solved via Normal Equations:

$$\mathbf{w}^* = (\mathbf{\Phi}^T\mathbf{\Phi})^{-1}\mathbf{\Phi}^T\mathbf{y}$$

> **Numerical Analysis Note:** As $d$ increases, the Vandermonde matrix becomes notoriously **ill-conditioned** — its condition number grows exponentially. Direct inversion becomes highly unstable, making Ridge regularization (Section 10) essential.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(2)
f_true  = lambda x: np.sin(2*np.pi*x)
n       = 15
x_train = np.sort(np.random.uniform(0, 1, n))
y_train = f_true(x_train) + np.random.normal(0, 0.2, n)
x_plot  = np.linspace(0, 1, 300)

def vandermonde(x, d):
    return np.column_stack([x**j for j in range(d+1)])

degrees_show = [1, 3, 6, 12]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, d in zip(axes.flat, degrees_show):
    Phi_tr   = vandermonde(x_train, d)
    Phi_plot = vandermonde(x_plot,  d)
    w        = np.linalg.lstsq(Phi_tr, y_train, rcond=None)[0]
    y_pred   = np.clip(Phi_plot @ w, -2.5, 2.5)
    mse_tr   = np.mean((y_train - Phi_tr@w)**2)
    cond     = np.linalg.cond(Phi_tr)
    ax.plot(x_plot, f_true(x_plot), 'k--', lw=2, alpha=0.7, label='$f^*$ (true)')
    ax.plot(x_plot, y_pred, '#E63946', lw=2.5, label=f'deg $d={d}$')
    ax.scatter(x_train, y_train, s=60, color='black', zorder=5, label='Training data')
    ax.set_xlim(-0.05,1.05); ax.set_ylim(-2.5,2.5)
    ax.set_title(f'$d={d}$ | Train MSE={mse_tr:.4f} | cond($\\Phi$)={cond:.1e}', fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Vandermonde Matrix & Polynomial Regression\nIll-conditioning grows exponentially with degree', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print('=== Condition Number of Vandermonde Matrix ===')
print(f'{"Degree":>8} | {"cond(Phi)":>15} | {"log10(cond)":>12}')
print('-'*42)
for d in range(1, 16):
    c = np.linalg.cond(vandermonde(x_train, d))
    print(f'{d:>8} | {c:>15.2e} | {np.log10(max(c,1e-20)):>12.2f}')


---
## 9. Mathematical Analysis and Diagnostics of Overfitting

When the polynomial degree $d$ is excessively high relative to the number of samples $n$, the polynomial gains too much geometric flexibility, driving training error to zero by interpolating the random noise $\epsilon$.

We track Loss (MSE) on **Training** and **Validation** subsets:

$$J_{\text{train}}(\mathbf{w}^*) = \frac{1}{n_{\text{train}}}\sum_{i\in\text{Train}}(y_i-\phi(x_i)^T\mathbf{w}^*)^2, \qquad J_{\text{val}}(\mathbf{w}^*) = \frac{1}{n_{\text{val}}}\sum_{i\in\text{Val}}(y_i-\phi(x_i)^T\mathbf{w}^*)^2$$

### The Divergent Behavior of Loss Curves

| Phase | $J_{\text{train}}$ | $J_{\text{val}}$ | Problem |
|-------|---------------------|------------------|---------|
| Low $d$ | $\gg 0$ | $\gg 0$ | Underfitting — model too rigid |
| Optimal $d$ | $\approx$ min | $\approx$ min | Good generalization |
| High $d$ | $\to 0$ | $\to \infty$ | Overfitting — memorizing noise |

Algebraically, when $d \geq n-1$ the system becomes determined/underdetermined and the polynomial passes exactly through every training point (**Runge's phenomenon**). This forces extreme oscillations between data points. While $J_{\text{train}} \to 0$, validation points — drawn from the same distribution but with independent noise — fall far from the wild artificial swings of the overfitted curve, making $J_{\text{val}}$ spike dramatically.

### Bias-Variance Decomposition

For the squared loss, the expected risk decomposes as:

$$\mathbb{E}_S[R(h_S)] = \underbrace{\sigma^2}_{\text{Noise}} + \underbrace{\text{Bias}^2(h)}_{\text{Approximation error}} + \underbrace{\text{Var}(h)}_{\text{Estimation error}}$$

Noise is irreducible. Bias decreases with model complexity; Variance increases. The optimal $d$ sits at the minimum of their sum.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
f_true  = lambda x: np.sin(2*np.pi*x)
N_total = 70
x_all   = np.sort(np.random.uniform(0,1,N_total))
y_all   = f_true(x_all) + np.random.normal(0, 0.25, N_total)

n_train  = 35
idx_tr   = np.random.choice(N_total, n_train, replace=False)
idx_vl   = np.setdiff1d(np.arange(N_total), idx_tr)
x_tr, y_tr = x_all[idx_tr], y_all[idx_tr]
x_vl, y_vl = x_all[idx_vl], y_all[idx_vl]

def vandermonde(x, d):
    return np.column_stack([x**j for j in range(d+1)])

max_deg = 18
tr_mses, vl_mses = [], []
for d in range(1, max_deg+1):
    Phi_tr = vandermonde(x_tr, d)
    Phi_vl = vandermonde(x_vl, d)
    w = np.linalg.lstsq(Phi_tr, y_tr, rcond=None)[0]
    tr_mses.append(np.mean((y_tr - Phi_tr@w)**2))
    vl_mses.append(np.mean((y_vl - Phi_vl@w)**2))

degrees  = np.arange(1, max_deg+1)
vl_clip  = np.clip(vl_mses, 0, 3.0)
best_d   = degrees[np.argmin(vl_clip)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(degrees, tr_mses,  '#2A9D8F', lw=2.5, marker='o', ms=5, label='$J_{\\text{train}}$')
ax.plot(degrees, vl_clip,  '#E63946', lw=2.5, marker='s', ms=5, label='$J_{\\text{val}}$ (clipped at 3)')
ax.axvline(best_d, color='gray', lw=2, linestyle='--', label=f'Optimal $d={best_d}$')
ax.fill_betweenx([0,3], 1, best_d-.5, alpha=0.07, color='blue',  label='Underfitting region')
ax.fill_betweenx([0,3], best_d+.5, max_deg, alpha=0.07, color='red', label='Overfitting region')
ax.set_xlabel('Polynomial degree $d$', fontsize=11)
ax.set_ylabel('MSE', fontsize=11)
ax.set_title('Train-Validation Loss: Bias-Variance Trade-off\n(Runge phenomenon for $d \\geq n-1$)', fontsize=10)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(0, 3)

ax2 = axes[1]
x_plot = np.linspace(-0.05, 1.05, 400)
ax2.plot(x_plot, f_true(x_plot), 'k--', lw=2, alpha=0.7, label='$f^*$ (true)')
ax2.scatter(x_tr, y_tr, s=40, color='black', zorder=5, alpha=0.7, label='Train')
for d, c, lbl in zip([1, best_d, 16],
                     ['#457B9D','#2A9D8F','#E63946'],
                     [f'deg 1 (underfit)', f'deg {best_d} (optimal)', 'deg 16 (overfit)']):
    Phi_d = vandermonde(x_tr, d)
    w = np.linalg.lstsq(Phi_d, y_tr, rcond=None)[0]
    y_p = np.clip(vandermonde(x_plot, d)@w, -3, 3)
    ax2.plot(x_plot, y_p, color=c, lw=2.2, label=lbl)
ax2.set_xlim(-0.05,1.05); ax2.set_ylim(-3,3)
ax2.set_title('Three Regimes Side by Side', fontsize=11)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.suptitle('Overfitting Diagnostics: Train-Validation Gap', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Optimal polynomial degree (min val MSE): d = {best_d}')


---
## 10. Advanced Spectral Analysis of Ridge Regularization ($L_2$)

### Mathematical Formulation and Convexity

To mitigate overfitting — a variance-driven phenomenon — we add a regularization term to the objective. In **Ridge Regression** ($L_2$ regularization):

$$J(\mathbf{w}) = \frac{1}{2}\sum_{i=1}^N(f(\mathbf{x}_i)-y_i)^2 + \frac{\lambda}{2}\|\mathbf{w}\|_2^2 = \frac{1}{2}(\mathbf{X}\mathbf{w}-\mathbf{y})^T(\mathbf{X}\mathbf{w}-\mathbf{y}) + \frac{\lambda}{2}\mathbf{w}^T\mathbf{w}$$

**Properties of the cost function:**
* **Strict Convexity:** Squaring ensures a strictly convex, globally differentiable landscape — unique global minimum guaranteed.
* **Isotropy of the $L_2$ Norm:** The constraint $\|\mathbf{w}\|_2^2 \leq t$ traces a **hypersphere** in parameter space. Being isotropic, it shrinks all weights uniformly toward zero, unlike the $L_1$ norm (Lasso) which forms a diamond-shaped polyhedron enforcing sparse solutions.

Setting $\nabla_\mathbf{w} J(\mathbf{w}) = \mathbf{0}$:

$$\mathbf{X}^T(\mathbf{X}\mathbf{w}-\mathbf{y}) + \lambda\mathbf{w} = \mathbf{0} \implies (\mathbf{X}^T\mathbf{X}+\lambda\mathbf{I})\mathbf{w} = \mathbf{X}^T\mathbf{y}$$

$$\boxed{\mathbf{w}_{\text{Ridge}} = (\mathbf{X}^T\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}}$$

---

### Spectral Analysis: Solving Ill-Conditioning via Eigenvalue Shifts

Since $\mathbf{X}^T\mathbf{X}$ is symmetric positive semi-definite, the **Spectral Theorem** gives:

$$\mathbf{X}^T\mathbf{X} = \mathbf{U}\mathbf{\Sigma}\mathbf{U}^T, \qquad \mathbf{\Sigma} = \text{diag}(\sigma_1^2,\sigma_2^2,\dots,\sigma_d^2)$$

Since $\mathbf{I} = \mathbf{U}\mathbf{I}\mathbf{U}^T$:

$$\mathbf{X}^T\mathbf{X}+\lambda\mathbf{I} = \mathbf{U}(\mathbf{\Sigma}+\lambda\mathbf{I})\mathbf{U}^T, \qquad \mathbf{\Sigma}_{\text{Ridge}} = \text{diag}(\sigma_1^2+\lambda,\dots,\sigma_d^2+\lambda)$$

**Numerical Implications:**
1. **Guaranteed Full Rank:** If $\sigma_i^2\approx 0$ (collinearity), adding $\lambda>0$ shifts every eigenvalue strictly above zero → $(\mathbf{X}^T\mathbf{X}+\lambda\mathbf{I})$ is **always invertible**.
2. **Selective Shrinkage:** The shrinkage factor $\frac{\sigma_i^2}{\sigma_i^2+\lambda}<1$ penalizes low-variance directions most.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
f_true  = lambda x: np.sin(2*np.pi*x)
n_train = 15
x_train = np.sort(np.random.uniform(0,1,n_train))
y_train = f_true(x_train) + np.random.normal(0, 0.25, n_train)
x_val   = np.sort(np.random.uniform(0,1,60))
y_val   = f_true(x_val) + np.random.normal(0, 0.25, 60)
x_plot  = np.linspace(-0.05, 1.05, 400)
DEGREE  = 12

def vandermonde(x, d):
    return np.column_stack([x**j for j in range(d+1)])

Phi_tr   = vandermonde(x_train, DEGREE)
Phi_vl   = vandermonde(x_val,   DEGREE)
Phi_plot = vandermonde(x_plot,  DEGREE)

lambdas  = [0, 1e-5, 1e-3, 1e-1, 1.0, 10.0]
colors_r = plt.cm.plasma(np.linspace(0.1, 0.9, len(lambdas)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(x_plot, f_true(x_plot), 'k--', lw=2, alpha=0.7, label='$f^*$')
ax.scatter(x_train, y_train, s=65, color='black', zorder=5)
for lam, color in zip(lambdas, colors_r):
    A = Phi_tr.T@Phi_tr + lam*np.eye(DEGREE+1)
    w = np.linalg.solve(A, Phi_tr.T@y_train)
    ax.plot(x_plot, np.clip(Phi_plot@w,-3,3), color=color, lw=1.9,
            label=f'$\\lambda={lam}$', alpha=0.9)
ax.set_xlim(-0.05,1.05); ax.set_ylim(-3,3)
ax.set_title(f'Ridge fits (degree {DEGREE}): effect of $\\lambda$', fontsize=11)
ax.legend(fontsize=7.5); ax.grid(True, alpha=0.3)

ax2 = axes[1]
lambda_range = np.logspace(-8, 2, 250)
tr_mses2, vl_mses2 = [], []
for lam in lambda_range:
    A = Phi_tr.T@Phi_tr + lam*np.eye(DEGREE+1)
    w = np.linalg.solve(A, Phi_tr.T@y_train)
    tr_mses2.append(np.mean((y_train - Phi_tr@w)**2))
    vl_mses2.append(np.mean((y_val   - Phi_vl@w)**2))
best_lam = lambda_range[np.argmin(np.clip(vl_mses2,0,1.5))]
ax2.semilogx(lambda_range, tr_mses2, '#2A9D8F', lw=2.2, label='$J_{\\text{train}}$')
ax2.semilogx(lambda_range, np.clip(vl_mses2,0,1.5), '#E63946', lw=2.2, label='$J_{\\text{val}}$')
ax2.axvline(best_lam, color='gray', lw=2, linestyle='--', label=f'Best $\\lambda\\approx{best_lam:.1e}$')
ax2.set_xlabel('$\\lambda$ (log scale)', fontsize=11)
ax2.set_ylabel('MSE', fontsize=11)
ax2.set_title('Train vs. Validation MSE as $\\lambda$ varies', fontsize=11)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3); ax2.set_ylim(0,1.5)

plt.suptitle('Ridge Regularization: From Overfitting to Regularized Fit', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Spectral plots
XtX     = Phi_tr.T @ Phi_tr
eigvals = np.sort(np.linalg.eigvalsh(XtX))[::-1]

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4.2))
ax3 = axes2[0]
ax3.bar(range(len(eigvals)), np.log10(np.maximum(eigvals,1e-20)),
        color='#457B9D', edgecolor='white', alpha=0.88)
ax3.set_xlabel('Eigenvalue index $i$', fontsize=11)
ax3.set_ylabel('$\\log_{10}(\\sigma_i^2)$', fontsize=11)
cond_nb = eigvals[0]/max(eigvals[-1],1e-20)
ax3.set_title(f'Eigenvalues of $\\mathbf{{X}}^T\\mathbf{{X}}$ (deg {DEGREE})\ncond = {cond_nb:.1e}', fontsize=10)
ax3.grid(True, alpha=0.3)

ax4 = axes2[1]
for lam in [0, 1e-4, 1e-2, 1.0]:
    ax4.semilogy(range(len(eigvals)), eigvals+lam, marker='o', ms=5, lw=1.8,
                 label=f'$\\sigma_i^2+\\lambda={lam}$')
ax4.set_xlabel('Eigenvalue index $i$', fontsize=11)
ax4.set_ylabel('$\\sigma_i^2+\\lambda$ (log scale)', fontsize=11)
ax4.set_title('Eigenvalue Shift: $\\lambda\\mathbf{I}$ ensures all $\\sigma_i^2+\\lambda>0$', fontsize=10)
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)
plt.suptitle('Spectral Analysis of Ridge Regularization', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'{"lambda":>10} | {"cond(XtX+lI)":>18} | {"Val MSE":>10}')
print('-'*44)
for lam in [0, 1e-6, 1e-4, 1e-2, 1e-1, 1.0]:
    A    = XtX + lam*np.eye(DEGREE+1)
    cond = np.linalg.cond(A)
    w    = np.linalg.solve(A, Phi_tr.T@y_train)
    vmse = np.mean((y_val - Phi_vl@w)**2)
    print(f'{lam:>10.1e} | {cond:>18.2e} | {min(vmse,99.9):>10.4f}')


---
## 11. Formal Derivation of the Expected Value of the Ridge Estimator: A Spectral Perspective

To understand exactly how $\lambda$ governs weight dynamics during learning, we evaluate $\mathbb{E}[\mathbf{w}_{\text{Ridge}}]$. This serves as our statistical compass to balance the Bias-Variance trade-off.

---

### 11.1 Connecting the Estimator to the Ground Truth

We assume targets are generated as $\mathbf{y} = \mathbf{X}\mathbf{w}_{\text{true}} + \boldsymbol{\epsilon}$, where $\mathbb{E}[\boldsymbol{\epsilon}]=\mathbf{0}$. Substituting into the Ridge solution:

$$\mathbf{w}_{\text{Ridge}} = (\mathbf{X}^T\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}^T(\mathbf{X}\mathbf{w}_{\text{true}}+\boldsymbol{\epsilon})$$

$$= (\mathbf{X}^T\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}^T\mathbf{X}\,\mathbf{w}_{\text{true}} + (\mathbf{X}^T\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}^T\boldsymbol{\epsilon}$$

Taking $\mathbb{E}[\cdot]$ — the noise term vanishes since $\mathbb{E}[\boldsymbol{\epsilon}]=\mathbf{0}$:

$$\mathbb{E}[\mathbf{w}_{\text{Ridge}}] = \left[(\mathbf{X}^T\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}^T\mathbf{X}\right]\mathbf{w}_{\text{true}}$$

---

### 11.2 Spectral (Eigendecomposition) View

Using $\mathbf{X}^T\mathbf{X}=\mathbf{U}\mathbf{\Sigma}\mathbf{U}^T$ and $\mathbf{I}=\mathbf{U}\mathbf{U}^T$:

$$\mathbb{E}[\mathbf{w}_{\text{Ridge}}] = \left[(\mathbf{U}\mathbf{\Sigma}\mathbf{U}^T+\lambda\mathbf{U}\mathbf{U}^T)^{-1}\mathbf{U}\mathbf{\Sigma}\mathbf{U}^T\right]\mathbf{w}_{\text{true}}$$

Factoring: $(\mathbf{U}(\mathbf{\Sigma}+\lambda\mathbf{I})\mathbf{U}^T)^{-1} = \mathbf{U}(\mathbf{\Sigma}+\lambda\mathbf{I})^{-1}\mathbf{U}^T$

$$\mathbb{E}[\mathbf{w}_{\text{Ridge}}] = \mathbf{U}(\mathbf{\Sigma}+\lambda\mathbf{I})^{-1}\underbrace{\mathbf{U}^T\mathbf{U}}_{=\mathbf{I}}\mathbf{\Sigma}\mathbf{U}^T\,\mathbf{w}_{\text{true}}$$

Since $(\mathbf{\Sigma}+\lambda\mathbf{I})^{-1}$ and $\mathbf{\Sigma}$ are both diagonal, their product is a single diagonal matrix:

$$\boxed{\mathbb{E}[\mathbf{w}_{\text{Ridge}}] = \mathbf{U}\operatorname{diag}\!\left(\frac{\sigma_1^2}{\sigma_1^2+\lambda},\frac{\sigma_2^2}{\sigma_2^2+\lambda},\dots,\frac{\sigma_d^2}{\sigma_d^2+\lambda}\right)\mathbf{U}^T\,\mathbf{w}_{\text{true}}}$$

---

### Key Insights

* **Selective Shrinkage:** $\mathbf{w}_{\text{true}}$ is rotated into the eigenspace ($\mathbf{U}^T$), scaled by shrinkage factors, then rotated back ($\mathbf{U}$).
* **Bias:** The factor $\frac{\sigma_j^2}{\sigma_j^2+\lambda}<1$ for all $\lambda>0$ proves Ridge introduces a deterministic **bias toward zero** — $\mathbb{E}[\mathbf{w}_{\text{Ridge}}]\neq\mathbf{w}_{\text{true}}$.
* **Numerical Stability:** If $\sigma_j^2\to 0$, OLS would explode ($\frac{1}{\sigma_j^2}\to\infty$). Ridge keeps the denominator bounded.
* **Limiting cases:** $\lambda\to 0$ → $\mathbb{E}[\mathbf{w}_{\text{Ridge}}]\to\mathbf{w}_{\text{true}}$ (OLS); $\lambda\to\infty$ → $\mathbb{E}[\mathbf{w}_{\text{Ridge}}]\to\mathbf{0}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Shrinkage factor curves
sigma2_values = np.logspace(-3, 2, 400)
lambdas_plot  = [0.0, 0.01, 0.1, 1.0, 10.0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for lam in lambdas_plot:
    shrink = sigma2_values / (sigma2_values + lam)
    lbl = f'$\\lambda={lam}$' if lam > 0 else '$\\lambda=0$ (OLS)'
    ax.semilogx(sigma2_values, shrink, lw=2.2, label=lbl)
ax.set_xlabel('Eigenvalue $\\sigma_j^2$ (log scale)', fontsize=11)
ax.set_ylabel('Shrinkage factor $\\frac{\\sigma_j^2}{\\sigma_j^2+\\lambda}$', fontsize=11)
ax.set_title('Shrinkage factor per eigendirection\n(small $\\sigma_j^2$ = penalized most)', fontsize=10)
ax.axhline(1, color='black', lw=1, linestyle=':', alpha=0.5)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)

# Numerical demo
d = 5
sigma2_true = np.array([10.0, 3.0, 1.0, 0.1, 0.01])
Sigma = np.diag(sigma2_true)
U, _  = np.linalg.qr(np.random.randn(d,d))
XtX_sim   = U @ Sigma @ U.T
w_true_sim = np.array([1.0, 0.8, 0.6, 0.4, 0.2])

ax2 = axes[1]
x_pos = np.arange(d)
ax2.bar(x_pos-0.32, w_true_sim, 0.2, color='black', label='$\\mathbf{w}_{\\text{true}}$', alpha=0.85)
offsets  = [-0.1, 0.1, 0.32]
lams_bar = [0.0, 0.1, 10.0]
clrs_bar = ['#2A9D8F','#457B9D','#E63946']
for lam, off, color in zip(lams_bar, offsets, clrs_bar):
    A    = XtX_sim + lam*np.eye(d)
    E_w  = np.linalg.inv(A) @ XtX_sim @ w_true_sim
    lbl  = f'$\\mathbb{{E}}[\\mathbf{{w}}_{{\\text{{Ridge}}}}],\\;\\lambda={lam}$'
    ax2.bar(x_pos+off, E_w, 0.2, color=color, label=lbl, alpha=0.85)
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'$\\sigma^2\\!={v}$' for v in sigma2_true], fontsize=9)
ax2.set_ylabel('Weight value', fontsize=11)
ax2.set_title('Shrinkage in action:\nlarger $\\lambda$ pushes $\\mathbb{E}[\\mathbf{w}]$ toward 0\n(strongest on low-$\\sigma^2$ directions)', fontsize=9.5)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.suptitle('Ridge Estimator Expected Value: Spectral Shrinkage Analysis', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print('=== Shrinkage factors per eigendirection ===')
print(f'{"sigma_j^2":>12} | {"lam=0 (OLS)":>12} | {"lam=0.1":>10} | {"lam=10":>10}')
print('-'*52)
for s2 in sigma2_true:
    row = [s2/(s2+lam) for lam in [0, 0.1, 10.0]]
    print(f'{s2:>12.3f} | {row[0]:>12.6f} | {row[1]:>10.6f} | {row[2]:>10.6f}')


---
## 12. Full Comparison: Linear vs. Polynomial vs. Ridge

Having built all three models from first principles, we now compare them systematically on the same dataset — examining fit quality, generalization, and the role of $\lambda$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

np.random.seed(2024)
f_true  = lambda x: 0.5*np.sin(3*np.pi*x) + x

# Dataset
N_train = 30
x_tr    = np.sort(np.random.uniform(0, 1, N_train))
y_tr    = f_true(x_tr) + np.random.normal(0, 0.15, N_train)
x_te    = np.sort(np.random.uniform(0, 1, 100))
y_te    = f_true(x_te) + np.random.normal(0, 0.15, 100)
x_plot  = np.linspace(-0.02, 1.02, 400)

# ── Models
models = {
    'Linear Regression\n($d=1$, no reg.)': make_pipeline(PolynomialFeatures(1), Ridge(alpha=0)),
    'Polynomial ($d=6$)\nno regularization': make_pipeline(PolynomialFeatures(6), Ridge(alpha=0)),
    'Polynomial ($d=6$)\n+ Ridge ($\\lambda=0.01$)': make_pipeline(PolynomialFeatures(6), Ridge(alpha=0.01)),
    'Polynomial ($d=12$)\n+ Ridge ($\\lambda=0.001$)': make_pipeline(PolynomialFeatures(12), Ridge(alpha=0.001)),
}
colors_m = ['#457B9D', '#E63946', '#2A9D8F', '#F4A261']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
results = []

for ax, (name, model), color in zip(axes.flat, models.items(), colors_m):
    model.fit(x_tr.reshape(-1,1), y_tr)
    y_pred_tr   = model.predict(x_tr.reshape(-1,1))
    y_pred_te   = model.predict(x_te.reshape(-1,1))
    y_pred_plot = np.clip(model.predict(x_plot.reshape(-1,1)), -1, 3)
    tr_mse  = mean_squared_error(y_tr, y_pred_tr)
    te_mse  = mean_squared_error(y_te, y_pred_te)
    results.append((name.replace('\\n',' '), tr_mse, te_mse))
    ax.plot(x_plot, f_true(x_plot), 'k--', lw=2, alpha=0.6, label='$f^*$ (true)')
    ax.scatter(x_tr, y_tr, s=45, color='black', zorder=5, alpha=0.8, label='Train')
    ax.plot(x_plot, y_pred_plot, color=color, lw=2.8, label='Model fit')
    ax.set_xlim(-0.02,1.02); ax.set_ylim(-0.5,2.0)
    ax.set_title(f'{name}\nTrain MSE={tr_mse:.4f}  |  Test MSE={te_mse:.4f}', fontsize=9.5)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.legend(fontsize=7.5); ax.grid(True, alpha=0.3)

plt.suptitle('Linear vs. Polynomial vs. Ridge Regression — Full Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Summary table
print('=== Summary Table ===')
print(f'{"Model":<45} | {"Train MSE":>10} | {"Test MSE":>10}')
print('-'*70)
for name, tr, te in results:
    print(f'{name:<45} | {tr:>10.4f} | {te:>10.4f}')


---
## Summary Table

| Concept | Key Formula | Core Insight |
|---------|-------------|-------------|
| Linear model | $\hat{y}=\mathbf{w}^T\mathbf{x}$ (augmented) | Affine map via bias trick |
| RSS Loss | $L=\|\mathbf{y}-\mathbf{X}\mathbf{w}\|_2^2$ | Convex, differentiable everywhere |
| ERM | $\hat{R}_N=\frac{1}{N}\sum_i L_i \xrightarrow{N\to\infty} R(f)$ | LLN approximation of true risk |
| Normal Equations | $\mathbf{w}^*=(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ | Exact in $\mathcal{O}(d^3)$ |
| Hat Matrix | $\mathbf{H}=\mathbf{X}(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T$, $\mathbf{H}^2=\mathbf{H}$, $\mathbf{H}^T=\mathbf{H}$ | Symmetric, idempotent projection |
| MLE $\equiv$ LSQ | $\arg\max\,\ell(\mathbf{w})=\arg\min\,\text{RSS}(\mathbf{w})$ | Under Gaussian noise |
| Gradient Descent | $\mathbf{w}^{(t+1)}=\mathbf{w}^{(t)}-\frac{\eta}{N}\mathbf{X}^T(\mathbf{X}\mathbf{w}^{(t)}-\mathbf{y})$ | $\mathcal{O}(Nd)$ per step, scalable |
| Vandermonde | $\mathbf{\Phi}_{ij}=x_i^{j-1}$, cond grows as $\mathcal{O}(e^d)$ | Nonlinear features, linear in $\mathbf{w}$ |
| Bias-Variance | $\mathbb{E}[R]=\sigma^2+\text{Bias}^2+\text{Var}$ | Fundamental trade-off |
| Ridge | $\mathbf{w}_{\text{Ridge}}=(\mathbf{X}^T\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$ | Eigenvalue shift → always invertible |
| Ridge shrinkage | $\mathbb{E}[\mathbf{w}_{\text{Ridge}}]=\mathbf{U}\operatorname{diag}\!\left(\frac{\sigma_j^2}{\sigma_j^2+\lambda}\right)\mathbf{U}^T\mathbf{w}_{\text{true}}$ | Biased but numerically stable |

---

**Next in the series →** `07_classification.ipynb`: Logistic Regression, Perceptron, Generalized Linear Models.
